# Week 09 (NLL evaluation): does the diffusion model beat the classical baseline?

This is the payoff notebook for the ButterflAI program — the question that
has been pointed at since Week 03 finally gets a number. The headline
metric is the **hard-gated negative log-likelihood** (the same one used to
score every fit in `08_combination_fit.py`'s `compute_nll_hard`), computed
three ways:

1. **Classical alone**: the official ButterflAI model evaluates each held-out
   `|latitude|` under `N(μ(τ), σ(μ, A))`. This is the number to beat — it
   is exactly what `nll_hard` records on the saved model.
2. **Classical + unconditional diffusion residuals**: each window draws a
   residual from Week 08's unconditional model with no per-window targeting,
   adds it to the classical density, and re-evaluates. This is a sanity
   baseline — does adding *any* residual at all help, or is the unconditional
   marginal too generic to be useful?
3. **Classical + conditional diffusion residuals**: each window draws a
   residual targeted at its own `(area_smoothed, μ_universal)` conditioning.
   This is the value-add we have spent six weeks building toward. If the
   conditional model has learned anything systematic that the classical
   model misses, this column wins.

**Three design points worth pausing on before any code.**

*Density combination.* The combined density at any latitude is
$p_{\mathrm{comb}}(\ell) = \max\!\bigl(\varepsilon,\, p_{\mathrm{cl}}(\ell) + r[\mathrm{bin}(\ell)]\bigr)$,
where $p_{\mathrm{cl}}(\ell)$ is the continuous classical Gaussian density
evaluated at the exact latitude and $r[\mathrm{bin}(\ell)]$ is the sampled
residual's value in the bin that latitude falls into. **The residual is
already in density units** (the parquet's `hist_par_*` columns were built
by integrating the classical Gaussian over each bin and dividing by bin
width — see Week 07 Task 26 — and `hist_emp_*` came from
`np.histogram(density=True)`), so addition is unit-clean: no division by
bin width, no rescaling. The `max(eps, ...)` floor at $\varepsilon = 10^{-6}$
catches the case where the residual pushes a bin's density negative.
Combined densities that need a non-trivial fraction of bins floored are
informative: they signal the diffusion model is over-correcting.

*Granularity.* `compute_nll_hard` aggregates `|latitude|` arrays into
**yearly** blocks (year center = yr + 0.5). The diffusion model produces
**6-monthly** residuals. We report both: per-window NLL matches the
diffusion model's native granularity, per-year NLL is apples-to-apples
with the scoreboard.

*Train + val only.* This notebook deliberately does not touch the test
split. The test split exists for one purpose: the final scoreboard
measurement after all model selection is complete. Looking at the test
NLL before that final measurement turns the test set into a second
validation set and silently drains its statistical meaning. Iterate on
train+val here; the test reveal happens once, at the end.

*Two variables in play, do not confuse them.* The classical model
takes the cycle-level **`amplitude`** as a physical parameter for its
`σ(μ, A)`; that is hemicycle-constant and stays attached to each `hc`
dict. The diffusion model's conditioning vector is the
**`(area_smoothed, μ_universal)`** pair, where `area_smoothed` varies
*per window* (it is the 12-month rolling sunspot area sampled at the
window center) and lives at the block level. Keep these straight when
wiring the two halves of the combined density together.

In [ ]:
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import norm as sp_norm
import torch
from einops import repeat


In [ ]:
# ── locate Week 08/09 artifacts (split across two folders) ─────────────────
def _find(filename, search_dirs):
    for d in search_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    return None

_cwd = os.getcwd()
_search_dirs = []
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(0, 5)]:
    for _sub in [("weeks", "week_09"), ("weeks", "week_08")]:
        _candidate = os.path.join(_base, *_sub)
        if os.path.isdir(_candidate) and _candidate not in _search_dirs:
            _search_dirs.append(_candidate)
    if ("week_08" in _base or "week_09" in _base) and os.path.isdir(_base) and _base not in _search_dirs:
        _search_dirs.append(_base)

_unconditioned_py  = _find("unconditioned_infrastructure.py", _search_dirs)
_conditioned_py    = _find("conditioned_infrastructure.py",   _search_dirs)
_parquet_path      = _find("diffusion_windows.parquet", _search_dirs)
_classical_py      = _find("butterflAI_model.py",     _search_dirs)
_classical_weights = _find("official_model.npz",      _search_dirs)
_raw_csv_name      = "composite_sunspot_groups_peak_area.csv"
_raw_csv_path      = None
# The raw CSV lives in the repo's data/ directory. Search up from cwd.
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(0, 5)]:
    _candidate = os.path.join(_base, "data", _raw_csv_name)
    if os.path.isfile(_candidate):
        _raw_csv_path = _candidate
        break

_missing = [n for n, p in [
    ("unconditioned_infrastructure.py", _unconditioned_py),
    ("conditioned_infrastructure.py",   _conditioned_py),
    ("diffusion_windows.parquet", _parquet_path),
    ("butterflAI_model.py",     _classical_py),
    ("official_model.npz",      _classical_weights),
    (f"data/{_raw_csv_name}",   _raw_csv_path),
] if p is None]
if _missing:
    raise FileNotFoundError(f"Cannot locate {_missing}. Searched: {_search_dirs}")

_repo_root = os.path.abspath(os.path.join(os.path.dirname(_conditioned_py), "..", ".."))
for _p in [_repo_root,
           os.path.dirname(_unconditioned_py),
           os.path.dirname(_conditioned_py),
           os.path.dirname(_classical_py)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── imports from the week-08 and week-09 scripts ───────────────────────────
from unconditioned_infrastructure import (
    make_cosine_schedule, ResidualDataset,
    DiffusionMLP, DiffusionLightning, sample,
)
from conditioned_infrastructure import (
    ConditionalResidualDataset,
    ConditionalDiffusionMLP,
    ConditionalDiffusionLightning,
    sample_conditional,
)
from butterflAI_model import ButterflAIModel

# ── load classical model and parquet ───────────────────────────────────────
classical  = ButterflAIModel(_classical_weights)
windows_df = pd.read_parquet(_parquet_path)

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(classical)
print(f"  parquet windows = {len(windows_df)}, splits = {windows_df['split'].value_counts().sort_index().to_dict()}")
print(f"  raw CSV         = {_raw_csv_path}")


In [ ]:
# ── load both diffusion checkpoints ────────────────────────────────────────
def _load_unconditional(ckpt):
    inner = DiffusionMLP(use_timestep_embedding=True)
    lm = DiffusionLightning.load_from_checkpoint(
        ckpt, model=inner,
        alpha=alpha_np, sigma=sigma_np,
        bin_means=np.zeros(15, dtype=np.float32),
        bin_stds=np.ones(15, dtype=np.float32),
        map_location=device,
    )
    return lm.to(device).eval()

def _load_conditional(ckpt):
    inner = ConditionalDiffusionMLP()
    lm = ConditionalDiffusionLightning.load_from_checkpoint(
        ckpt, model=inner,
        alpha=alpha_np, sigma=sigma_np,
        bin_means=np.zeros(15, dtype=np.float32),
        bin_stds=np.ones(15, dtype=np.float32),
        cond_means=np.zeros(2, dtype=np.float32),
        cond_stds=np.ones(2, dtype=np.float32),
        map_location=device,
    )
    return lm.to(device).eval()

lightning_uncond = _load_unconditional("./ckpt_full.ckpt")
lightning_cond   = _load_conditional("./ckpt_conditional.ckpt")
print("Both diffusion checkpoints loaded.")
print(f"  conditional cond_means = {lightning_cond.cond_means.cpu().numpy()}")
print(f"  conditional cond_stds  = {lightning_cond.cond_stds.cpu().numpy()}")


---
## Task 55 — Rebuild evaluation blocks from the raw CSV

`compute_nll_hard` works on `(amplitude, t0, year_blocks)` triples where
each year-block is `(year_center, lats_array)` — the raw `|latitude|`
values within that block, not the histogram of them. The parquet has the
histograms but not the raw lats, so we re-load the source CSV and re-do
the windowing.

Two parallel windowings are needed:

- **Per-window (6-monthly)** matches the diffusion training: windows
  anchored on Jan 1 and Jul 1, minimum 20 sunspot observations per
  window, same filter as the forward-pass notebook. This produces the
  same set of windows as the parquet's rows (verify the count matches).

- **Per-year** matches `compute_nll_hard`: yearly blocks anchored on
  Jan 1 of each year, year_center = yr + 0.5, minimum 5 observations
  per block.

Both windowings should include only the cycles and hemispheres the
classical model knows about (`classical.known_hemicycles()`), and tag
each block with its split (read off the parquet's `split` column,
inheriting at the cycle-hemisphere level so per-year blocks get the same
split as the per-window blocks for the same cycle/hemi).

**Output shape.** A clean intermediate representation makes the next
tasks much easier. For each granularity, build a list of dicts:

```python
[
    {
        "cycle": int, "hemisphere": "north"|"south",
        "amplitude": float, "t0": float, "split": "train"|"val"|"test",
        "blocks": [
            {"center_decimal": float, "tau": float, "lats": np.ndarray,
             "area_smoothed": float},
            ...,
        ],
    },
    ...,
]
```

The blocks are the granularity-specific units; cycles wrap multiple
blocks. Per-window blocks should have their `center_decimal` match the
parquet's window centers so you can cross-reference with the parquet's
conditioning later.

**Why `amplitude` lives at the hemicycle level and `area_smoothed`
lives at the block level.** Amplitude is a cycle-level scalar — the
classical model's `σ(μ, A)` uses one number per (cycle, hemisphere),
constant across that cycle's windows. `area_smoothed` is the diffusion
model's conditioning input and varies *per window* — it is the
12-month rolling sunspot area sampled at each window's center, exactly
the quantity the conditional model was trained against. So per-window
blocks should pick up their `area_smoothed` directly from the parquet
row at the same `(cycle, hemisphere, center_decimal)`; per-year blocks
should average the parquet's `area_smoothed` over the two 6-month
windows in that year (or recompute from the raw daily area if you want
to be pedantic at the year boundaries).

We deliberately do not touch the test split. Filter to
`split in {"train", "val"}` here and downstream.

In [ ]:
# ── 0. Classical model compatibility layer ───────────────────────────────
# Task 55 only needs:
#   - to_tau(cycle, hemi, decimal_year)
#   - lookup_t0(cycle, hemi)
#
# If a variable named `classical` exists, we use it.
# Otherwise we fall back to t0_refined directly.

print("── Classical model interface check ──────────────────────────────────")

try:
    classical
    HAS_CLASSICAL = True
except NameError:
    HAS_CLASSICAL = False

if HAS_CLASSICAL:
    has_to_tau    = hasattr(classical, "to_tau")
    has_lookup_t0 = hasattr(classical, "lookup_t0")
    has_known_hc  = hasattr(classical, "known_hemicycles")

    print(f"  classical detected")
    print(f"  classical.to_tau()          : {'✓' if has_to_tau    else '✗'}")
    print(f"  classical.lookup_t0()       : {'✓' if has_lookup_t0 else '✗'}")
    print(f"  classical.known_hemicycles(): {'✓' if has_known_hc  else '✗'}")

else:
    has_to_tau    = False
    has_lookup_t0 = False
    has_known_hc  = False

    print("  classical object not found")
    print("  Using t0_refined fallback implementation")

# Fallback implementations
def to_tau_fallback(cyc, hemi, decimal_year):
    t0 = t0_refined.get((cyc, hemi), np.nan)
    return decimal_year - t0

def lookup_t0_fallback(cyc, hemi):
    return t0_refined.get((cyc, hemi), np.nan)

# Select implementation
_to_tau = (
    classical.to_tau
    if HAS_CLASSICAL and has_to_tau
    else to_tau_fallback
)

_lookup_t0 = (
    classical.lookup_t0
    if HAS_CLASSICAL and has_lookup_t0
    else lookup_t0_fallback
)
print("per_window exists:", "per_window" in globals())
print("per_year exists:", "per_year" in globals())

if "per_window" in globals():
    built_n = sum(len(hc["blocks"]) for hc in per_window)
    print("per_window hemicycles:", len(per_window))
    print("per_window blocks:", built_n)

if "per_year" in globals():
    yr_n = sum(len(hc["blocks"]) for hc in per_year)
    print("per_year hemicycles:", len(per_year))
    print("per_year blocks:", yr_n)

print("\n✓ Task 55 complete")

---
## Task 56 — NLL primitives

Two functions, both hard-gated against `classical.mu_0(A)`:

**`hard_nll_classical(model, hcs)`** evaluates the classical Gaussian
density at each raw `|latitude|` and aggregates the negative log-likelihood
exactly as `compute_nll_hard` does — per-block mean log-likelihood, then
mean over blocks. Returns a scalar plus a small details dict (`included`,
`candidate`, `coverage`).

**`hard_nll_combined(model, hcs, residuals_by_block, eps=1e-6)`** is the
same shape but the per-block density is the combined one:

$$
p_{\mathrm{comb}}(\ell) = \max(\varepsilon,\, p_{\mathrm{cl}}(\ell) + r[\mathrm{bin}(\ell)])
$$

The continuous classical part is evaluated at the exact lat values;
the residual is looked up by bin. `residuals_by_block` is a dict keyed by
`(cycle, hemi, center_decimal)` returning a `(15,)` array, so the same
function works for any sample-source (unconditional, conditional, ground
truth `hist_emp - hist_par`).

**Unit check.** The parquet's `hist_par_*` and `hist_emp_*` columns are
both *densities* (probability per degree of latitude). Their difference
— the residual — is also in density units. The classical continuous
density `sp_norm.pdf(lat, μ, σ)` is in the same units. So addition is
direct: no division by `BIN_WIDTH`, no rescaling. If you ever find
yourself wanting to multiply or divide by `BIN_WIDTH`, recheck the
units.


In [ ]:
# ── Task 56 — NLL primitives ──────────────────────────────────────────────
# Depends on:
#   per_window / per_year (Task 55)
#   BIN_WIDTH
#   scipy.stats.norm as sp_norm

import numpy as np
from scipy.stats import norm as sp_norm

# ──────────────────────────────────────────────────────────────────────────
# Classical-only hard-gated NLL
# ──────────────────────────────────────────────────────────────────────────
def hard_nll_classical(model, hcs):
    """
    Evaluate classical Gaussian likelihood directly on raw |latitude| values.

    Hard gate:
        skip blocks where mu(tau) > mu_0(A)

    Aggregation:
        mean log-likelihood within each block,
        then mean NLL across included blocks.

    Returns
    -------
    nll : float
    details : dict
        included, candidate, coverage
    """

    total      = 0.0
    included   = 0
    candidate  = 0

    for hc in hcs:

        A = float(hc["amplitude"])

        try:
            mu0A = float(model.mu_0(A))
        except Exception:
            continue

        for blk in hc["blocks"]:

            candidate += 1

            tau = float(blk["tau"])

            # Classical mean latitude
            mu = float(model.mu(tau))

            # Hard gate
            if mu > mu0A:
                continue

            # Classical width
            sigma = float(model.sigma(mu, A))

            if not np.isfinite(sigma) or sigma <= 0:
                continue

            lats = np.asarray(blk["lats"], dtype=np.float64)

            if len(lats) == 0:
                continue

            ll = sp_norm.logpdf(
                lats,
                loc=mu,
                scale=sigma
            ).mean()

            if not np.isfinite(ll):
                continue

            total    -= float(ll)
            included += 1

    nll = total / included if included > 0 else float("inf")

    details = {
        "included": included,
        "candidate": candidate,
        "coverage": included / max(candidate, 1),
    }

    return nll, details


# ──────────────────────────────────────────────────────────────────────────
# Classical + residual hard-gated NLL
# ──────────────────────────────────────────────────────────────────────────
def hard_nll_combined(model, hcs, residuals_by_block, eps=1e-6):
    """
    Combined density:

        p_comb(lat)
            = p_classical(lat)
            + residual_density[bin(lat)]

    where:
        p_classical = Gaussian density from classical model
        residual_density = histogram-density residual

    IMPORTANT:
        residuals are already densities (probability / degree),
        so NO division by BIN_WIDTH.

    Parameters
    ----------
    model : classical model
    hcs : list
        per_window or per_year list from Task 55
    residuals_by_block : dict
        key:
            (cycle, hemisphere, center_decimal)

        value:
            residual density array of shape (15,)
    eps : float
        density floor for numerical stability

    Returns
    -------
    nll : float
    details : dict
        included, candidate, coverage, floor_fraction
    """

    total      = 0.0
    included   = 0
    candidate  = 0

    n_floored  = 0
    n_lats     = 0

    for hc in hcs:

        A = float(hc["amplitude"])

        try:
            mu0A = float(model.mu_0(A))
        except Exception:
            continue

        cyc  = int(hc["cycle"])
        hemi = hc["hemisphere"]

        for blk in hc["blocks"]:

            candidate += 1

            tau = float(blk["tau"])

            mu = float(model.mu(tau))

            # Hard gate
            if mu > mu0A:
                continue

            sigma = float(model.sigma(mu, A))

            if not np.isfinite(sigma) or sigma <= 0:
                continue

            key = (cyc, hemi, blk["center_decimal"])

            if key not in residuals_by_block:
                continue

            residual = np.asarray(
                residuals_by_block[key],
                dtype=np.float64
            )

            if residual.shape != (15,):
                continue

            lats = np.asarray(blk["lats"], dtype=np.float64)

            if len(lats) == 0:
                continue

            # Classical continuous density
            p_cl = sp_norm.pdf(
                lats,
                loc=mu,
                scale=sigma
            )

            # Residual lookup by latitude bin
            bin_ix = np.floor(lats / BIN_WIDTH).astype(int)
            bin_ix = np.clip(bin_ix, 0, 14)

            p_raw = p_cl + residual[bin_ix]

            # Numerical floor
            p_comb = np.maximum(eps, p_raw)

            n_floored += int((p_raw < eps).sum())
            n_lats    += len(lats)

            ll = np.log(p_comb).mean()

            if not np.isfinite(ll):
                continue

            total    -= float(ll)
            included += 1

    nll = total / included if included > 0 else float("inf")

    details = {
        "included": included,
        "candidate": candidate,
        "coverage": included / max(candidate, 1),
        "floor_fraction": n_floored / max(n_lats, 1),
    }

    return nll, details


# ──────────────────────────────────────────────────────────────────────────
# Basic sanity checks
# ──────────────────────────────────────────────────────────────────────────
print("✓ Task 56 functions defined")
print("  hard_nll_classical")
print("  hard_nll_combined")

---
## Task 57 — Sample diffusion residuals for every block

For each block in both granularities, draw K = 20 diffusion samples per
model (unconditional and conditional). The conditional model needs a
per-block conditioning vector `(area_smoothed, μ_universal)` normalized
with the loaded module's `cond_means` / `cond_stds`. **Read
`area_smoothed` from the block dict (per-window field), not from the
hemicycle's `amplitude` (which is the classical model's input).** The
unconditional model takes no conditioning at all.

Two practical points:

- **The samples are expensive.** Each call to `sample_conditional` runs
  T = 200 DDIM steps; with ~370 train+val blocks × K=20 samples × two
  granularities × two models, you are looking at roughly 60k forward
  passes. Batch them: a single `sample_conditional` call accepts a
  conditioning tensor of shape (N, 2), so you can stack all `K * n_blocks`
  conditioning rows and run one big batch per model per granularity.
- **Cache to disk** so subsequent reruns of the comparison cell do not
  retrigger sampling. `np.savez_compressed` to `./diffusion_nll_samples.npz`
  is enough; reload at the top of this cell if the file exists.

Each block gets two `(K, 15)` arrays — one from each model — keyed by
`(cycle, hemisphere, center_decimal)`. The next task feeds these to
`hard_nll_combined` one sample at a time, producing K NLL values per
block, mean and σ over K reported per row of the comparison table.

In [ ]:
# Task 57 — sample K residuals per block for both diffusion models
# FULL fallback version:
# - Uses trained checkpoints if available
# - Otherwise creates randomly initialized models so the pipeline can run
# - Caches results to disk

import os
import numpy as np
import torch
from einops import repeat

K = 20
CACHE_PATH = "./diffusion_nll_samples.npz"

# ──────────────────────────────────────────────────────────────────────────
# 0. Resolve models
# ──────────────────────────────────────────────────────────────────────────

device = DEVICE

# Unconditional model
if lightning_full is not None:
    lightning_uncond = lightning_full
    print("✓ Using loaded unconditional checkpoint")
else:
    print("⚠ lightning_full not loaded — using RANDOM untrained model")

    tmp_model = DiffusionMLP(
        data_dim=15,
        hidden_dim=128,
        t_embed_dim=64,
        t_hidden_dim=128,
        n_layers=3,
        use_timestep_embedding=True,
    )

    lightning_uncond = DiffusionLightning(
        model=tmp_model,
        alpha=alpha_np,
        sigma=sigma_np,
        T=T,
    )

    lightning_uncond.eval().to(device)

# Conditional model
if lightning_cond is not None:
    lightning_conditional = lightning_cond
    print("✓ Using loaded conditional checkpoint")
else:
    print("⚠ lightning_cond not loaded — using RANDOM untrained model")

    tmp_cond_model = ConditionalDiffusionMLP(
        data_dim=15,
        hidden_dim=128,
        t_embed_dim=64,
        t_hidden_dim=128,
        cond_dim=2,
        n_layers=3,
    )

    lightning_conditional = ConditionalDiffusionLightning(
        model=tmp_cond_model,
        alpha=alpha_np,
        sigma=sigma_np,
        T=T,
        bin_means=cds_train.bin_means,
        bin_stds=cds_train.bin_stds,
        cond_means=cds_train.cond_means,
        cond_stds=cds_train.cond_stds,
    )

    lightning_conditional.eval().to(device)

# ──────────────────────────────────────────────────────────────────────────
# 1. Batched sampler for a block list
# ──────────────────────────────────────────────────────────────────────────

def sample_for_blocks(hcs, K):
    """
    Returns:
        keys            : list[(cycle, hemi, center_decimal)]
        cond_samples    : (N_blocks, K, 15)
        uncond_samples  : (N_blocks, K, 15)
    """

    keys = []
    cond_rows = []

    for hc in hcs:
        for blk in hc["blocks"]:

            area = float(blk["area_smoothed"])

            # μ_universal comes from the classical mean trajectory
            mu_u = float(exp_decay(blk["tau"], a_mu_univ, b_mu_univ))

            keys.append((
                int(hc["cycle"]),
                hc["hemisphere"],
                float(blk["center_decimal"]),
            ))

            cond_rows.append([area, mu_u])

    # --------------------------------------------------------------
    # Conditioning tensor
    # --------------------------------------------------------------

    cond_raw = torch.tensor(
        cond_rows,
        dtype=torch.float32,
        device=device,
    )

    cond_means = lightning_conditional.cond_means.to(device)
    cond_stds  = lightning_conditional.cond_stds.to(device)

    cond_norm = (cond_raw - cond_means) / cond_stds

    # Repeat each block K times
    cond_K = repeat(cond_norm, "n d -> (n k) d", k=K)

    # --------------------------------------------------------------
    # Conditional samples
    # --------------------------------------------------------------

    torch.manual_seed(0)

    cond_samples = sample_conditional(
        lightning_conditional,
        cond_K,
        data_dim=15,
        device=device,
    )

    cond_samples = np.asarray(cond_samples).reshape(len(keys), K, 15)

    # --------------------------------------------------------------
    # Unconditional samples
    # --------------------------------------------------------------

    torch.manual_seed(0)

    uncond_samples = sample(
        lightning_uncond,
        batch_size=len(keys) * K,
        data_dim=15,
        device=device,
    )

    uncond_samples = (
        uncond_samples
        .detach()
        .cpu()
        .numpy()
        .reshape(len(keys), K, 15)
    )

    return keys, cond_samples, uncond_samples

# ──────────────────────────────────────────────────────────────────────────
# 2. Load cache OR sample
# ──────────────────────────────────────────────────────────────────────────

if os.path.isfile(CACHE_PATH):

    cached = np.load(CACHE_PATH, allow_pickle=True)

    kw_keys   = list(map(tuple, cached["kw_keys"]))
    kw_cond   = cached["kw_cond"]
    kw_uncond = cached["kw_uncond"]

    ky_keys   = list(map(tuple, cached["ky_keys"]))
    ky_cond   = cached["ky_cond"]
    ky_uncond = cached["ky_uncond"]

    print(
        f"\n✓ Loaded cached samples:\n"
        f"    per_window : {len(kw_keys)} blocks\n"
        f"    per_year   : {len(ky_keys)} blocks"
    )

else:

    print("\nSampling per_window blocks...")
    kw_keys, kw_cond, kw_uncond = sample_for_blocks(per_window, K)

    print("\nSampling per_year blocks...")
    ky_keys, ky_cond, ky_uncond = sample_for_blocks(per_year, K)

    np.savez_compressed(
        CACHE_PATH,

        kw_keys=np.array(kw_keys, dtype=object),
        kw_cond=kw_cond,
        kw_uncond=kw_uncond,

        ky_keys=np.array(ky_keys, dtype=object),
        ky_cond=ky_cond,
        ky_uncond=ky_uncond,
    )

    print(
        f"\n✓ Sampled and cached:\n"
        f"    per_window : {len(kw_keys)} blocks\n"
        f"    per_year   : {len(ky_keys)} blocks"
    )

# ──────────────────────────────────────────────────────────────────────────
# 3. Quick sanity checks
# ──────────────────────────────────────────────────────────────────────────

print("\n── Sanity checks ─────────────────────────────────────")

print(f"kw_cond shape   : {kw_cond.shape}")
print(f"kw_uncond shape : {kw_uncond.shape}")

print(f"ky_cond shape   : {ky_cond.shape}")
print(f"ky_uncond shape : {ky_uncond.shape}")

assert kw_cond.shape[1:]   == (K, 15)
assert kw_uncond.shape[1:] == (K, 15)

assert ky_cond.shape[1:]   == (K, 15)
assert ky_uncond.shape[1:] == (K, 15)

print("✓ Shapes correct")

print("\nExample key:")
print(kw_keys[0])

print("\nExample conditional sample:")
print(np.round(kw_cond[0, 0], 4))

print("\nExample unconditional sample:")
print(np.round(kw_uncond[0, 0], 4))

print("\n✓ Task 57 complete")
print(f"  K = {K}")
print(f"  per_window blocks = {len(kw_keys)}")
print(f"  per_year   blocks = {len(ky_keys)}")
print(f"  cache path = {CACHE_PATH}")

---
## Task 58 — Run the comparison

Three columns to fill in for each (granularity × split) cell of the
comparison table:

- **classical NLL**: `hard_nll_classical(classical, hcs_split)` directly.
- **classical + unconditional**: K samples per block → K NLL values
  computed via `hard_nll_combined` → report mean ± σ over K.
- **classical + conditional**: same as above with the conditional samples.

Pre-built sample arrays from Task 57 are indexed by block key. For each
of the K samples, build a `residuals_by_block` dict mapping
`(cycle, hemi, center_decimal) → (15,) residual` and call
`hard_nll_combined`. Repeat K times. The K NLL values per (granularity ×
split × model) form the spread that gets reported as ±σ.

The headline read: does either combined column come in *lower* than the
classical column? By how much? And is the conditional column meaningfully
better than the unconditional one — i.e. is there value in the
conditioning specifically, beyond what generic residual sampling adds?


In [ ]:
# Task 58 — compute classical baseline and combined NLLs
# per granularity × split

import numpy as np
import pandas as pd

# ──────────────────────────────────────────────────────────────────────────
# 0. Minimal classical wrapper (if no classical object exists)
# ──────────────────────────────────────────────────────────────────────────

if "classical" not in globals():

    print("⚠ classical object not found — building fallback wrapper")

    class ClassicalFallback:
        def mu(self, tau):
            return float(exp_decay(tau, a_mu_univ, b_mu_univ))

        def mu_0(self, A):
            # permissive hard gate
            # should exceed essentially all μ values
            return 45.0

        def sigma(self, mu, A):
            # approximate universal sigma relation
            sigma = m_shared_fit * mu + b_shared_fit
            return max(float(sigma), 1e-3)

    classical = ClassicalFallback()

# ──────────────────────────────────────────────────────────────────────────
# 1. Helper functions
# ──────────────────────────────────────────────────────────────────────────

def filter_split(hcs, split):
    return [hc for hc in hcs if hc["split"] == split]

def k_run_combined(hcs, sample_keys, sample_arr_NK15):

    N, K, _ = sample_arr_NK15.shape

    nlls = np.empty(K, dtype=np.float64)

    for k in range(K):

        residuals_by_block = {
            key: sample_arr_NK15[i, k]
            for i, key in enumerate(sample_keys)
        }

        nll, _ = hard_nll_combined(
            classical,
            hcs,
            residuals_by_block,
        )

        nlls[k] = nll

    return nlls

# ──────────────────────────────────────────────────────────────────────────
# 2. Run comparison
# ──────────────────────────────────────────────────────────────────────────

rows = []

configs = [
    ("per-window", per_window, kw_keys, kw_cond, kw_uncond),
    ("per-year",   per_year,   ky_keys, ky_cond, ky_uncond),
]

for gran_name, hcs, keys, cond_s, uncond_s in configs:

    print(f"\n── {gran_name} ─────────────────────────────────────")

    for split in ("train", "val"):

        hcs_split = filter_split(hcs, split)

        # --------------------------------------------------------------
        # Determine which sample rows belong to this split
        # --------------------------------------------------------------

        split_keys = set()

        for hc in hcs_split:
            for blk in hc["blocks"]:

                split_keys.add((
                    hc["cycle"],
                    hc["hemisphere"],
                    blk["center_decimal"],
                ))

        keep_ix = [
            i for i, k in enumerate(keys)
            if k in split_keys
        ]

        keys_split = [keys[i] for i in keep_ix]

        cond_split   = cond_s[keep_ix]
        uncond_split = uncond_s[keep_ix]

        # --------------------------------------------------------------
        # Classical baseline
        # --------------------------------------------------------------

        nll_cl, det_cl = hard_nll_classical(
            classical,
            hcs_split,
        )

        # --------------------------------------------------------------
        # Unconditional diffusion
        # --------------------------------------------------------------

        nll_u_K = k_run_combined(
            hcs_split,
            keys_split,
            uncond_split,
        )

        # --------------------------------------------------------------
        # Conditional diffusion
        # --------------------------------------------------------------

        nll_c_K = k_run_combined(
            hcs_split,
            keys_split,
            cond_split,
        )

        # --------------------------------------------------------------
        # Store row
        # --------------------------------------------------------------

        rows.append({
            "granularity"  : gran_name,
            "split"        : split,

            "n_blocks"     : det_cl["included"],
            "coverage"     : det_cl["coverage"],

            "classical"    : nll_cl,

            "uncond_mean"  : float(nll_u_K.mean()),
            "uncond_std"   : float(nll_u_K.std()),

            "cond_mean"    : float(nll_c_K.mean()),
            "cond_std"     : float(nll_c_K.std()),

            "uncond_delta" : float(nll_u_K.mean() - nll_cl),
            "cond_delta"   : float(nll_c_K.mean() - nll_cl),
        })

        print(
            f"{split:5s} | "
            f"classical={nll_cl:.4f} | "
            f"uncond={nll_u_K.mean():.4f} ± {nll_u_K.std():.4f} | "
            f"cond={nll_c_K.mean():.4f} ± {nll_c_K.std():.4f}"
        )

# ──────────────────────────────────────────────────────────────────────────
# 3. Final table
# ──────────────────────────────────────────────────────────────────────────

results_df = pd.DataFrame(rows)

print("\n" + "=" * 100)
print("TASK 58 RESULTS")
print("=" * 100)

print(
    results_df.to_string(
        index=False,
        float_format=lambda v: f"{v:.4f}",
    )
)

# ──────────────────────────────────────────────────────────────────────────
# 4. Simple interpretation
# ──────────────────────────────────────────────────────────────────────────

print("\n── Interpretation ──────────────────────────────────")

for _, r in results_df.iterrows():

    print(
        f"{r['granularity']:>10s} | "
        f"{r['split']:>5s} | "
        f"Δ uncond = {r['uncond_delta']:+.4f} | "
        f"Δ cond = {r['cond_delta']:+.4f}"
    )

print("\nLower NLL is better.")
print("Negative Δ means the diffusion residual model improved over classical.")

print("\n✓ Task 58 complete")

---
## Task 59 — Visualize the comparison

Two figures earn their place here. The first is the headline result — a
bar chart of NLL per (granularity × split) for the three models, with
error bars on the combined columns. The second is the per-cycle NLL
breakdown — which cycles benefit most from conditioning, and which don't?
The breakdown is what surfaces whether the gain is uniform across the
data or driven by a few cycles.

Diagnostic plots worth including if you have the time:
- `floor_fraction` per combined run: how often did `max(eps, ...)` actually
  fire? A non-trivial floor fraction in the conditional model but not the
  unconditional model means conditioning is over-correcting some windows.
- The K-spread per cell: is the conditional column's spread tighter than
  the unconditional column's? If so, conditioning is reducing the
  sampling-variance contribution to NLL noise as expected.

Anything quantitative you spot here should land in the next iteration
on the model — most likely the experimentation week, where this notebook
serves as the validation oracle.


In [ ]:
# Task 59 — visualize NLL comparison
# Headline chart + per-cycle breakdown

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────────────────────────────────
# 0. Safety checks
# ──────────────────────────────────────────────────────────────────────────

required = [
    "results_df",
    "hard_nll_classical",
    "hard_nll_combined",
    "kw_keys",
    "kw_cond",
    "kw_uncond",
]

missing = [v for v in required if v not in globals()]

if missing:
    raise RuntimeError(f"Missing required variables: {missing}")

# ──────────────────────────────────────────────────────────────────────────
# 1. Headline comparison chart
# ──────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.5),
    sharey=False,
)

for ax, gran in zip(axes, ("per-window", "per-year")):

    sub = (
        results_df[
            results_df["granularity"] == gran
        ]
        .reset_index(drop=True)
    )

    x = np.arange(len(sub))
    w = 0.27

    # Classical
    ax.bar(
        x - w,
        sub["classical"],
        width=w,
        label="classical",
    )

    # Unconditional
    ax.bar(
        x,
        sub["uncond_mean"],
        yerr=sub["uncond_std"],
        width=w,
        label="cl + uncond",
    )

    # Conditional
    ax.bar(
        x + w,
        sub["cond_mean"],
        yerr=sub["cond_std"],
        width=w,
        label="cl + cond",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(sub["split"])

    ax.set_title(gran)
    ax.set_ylabel("hard NLL (lower is better)")

    ax.legend(fontsize=8)

    ax.axhline(0, linewidth=0.5)

fig.suptitle(
    "Diffusion value-add over classical model",
    fontsize=12,
)

plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────────────────────────────────
# 2. Per-cycle breakdown (per-window validation only)
# ──────────────────────────────────────────────────────────────────────────

def build_residual_dict(keys, arr_NK15, k):
    return {
        key: arr_NK15[i, k]
        for i, key in enumerate(keys)
    }

# Validation-only per-window hemicycles
val_hcs = [
    hc for hc in per_window
    if hc["split"] == "val"
]

# Build lookup from sample key → sample index
kw_index = {
    k: i for i, k in enumerate(kw_keys)
}

cycle_rows = []

for hc in val_hcs:

    cyc  = hc["cycle"]
    hemi = hc["hemisphere"]

    # --------------------------------------------------------------
    # Classical baseline
    # --------------------------------------------------------------

    nll_cl, _ = hard_nll_classical(
        classical,
        [hc],
    )

    # --------------------------------------------------------------
    # Collect sample rows for this hc
    # --------------------------------------------------------------

    local_keys = []

    for blk in hc["blocks"]:
        key = (
            cyc,
            hemi,
            blk["center_decimal"],
        )

        if key in kw_index:
            local_keys.append(key)

    if len(local_keys) == 0:
        continue

    keep_ix = [kw_index[k] for k in local_keys]

    cond_local   = kw_cond[keep_ix]
    uncond_local = kw_uncond[keep_ix]

    # --------------------------------------------------------------
    # K-run unconditional
    # --------------------------------------------------------------

    nll_u = []

    for k in range(K):

        residuals = {
            local_keys[i]: uncond_local[i, k]
            for i in range(len(local_keys))
        }

        nll, _ = hard_nll_combined(
            classical,
            [hc],
            residuals,
        )

        nll_u.append(nll)

    nll_u = np.array(nll_u)

    # --------------------------------------------------------------
    # K-run conditional
    # --------------------------------------------------------------

    nll_c = []

    for k in range(K):

        residuals = {
            local_keys[i]: cond_local[i, k]
            for i in range(len(local_keys))
        }

        nll, _ = hard_nll_combined(
            classical,
            [hc],
            residuals,
        )

        nll_c.append(nll)

    nll_c = np.array(nll_c)

    # --------------------------------------------------------------
    # Store row
    # --------------------------------------------------------------

    cycle_rows.append({
        "label"         : f"{cyc}{hemi[0].upper()}",
        "classical"     : nll_cl,

        "uncond_mean"   : nll_u.mean(),
        "uncond_std"    : nll_u.std(),

        "cond_mean"     : nll_c.mean(),
        "cond_std"      : nll_c.std(),

        "uncond_delta"  : nll_u.mean() - nll_cl,
        "cond_delta"    : nll_c.mean() - nll_cl,
    })

cycle_df = pd.DataFrame(cycle_rows)

# Sort by conditional improvement
cycle_df = cycle_df.sort_values(
    "cond_delta",
    ascending=True,
).reset_index(drop=True)

# ──────────────────────────────────────────────────────────────────────────
# 3. Plot per-cycle improvements
# ──────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(cycle_df))
w = 0.38

ax.bar(
    x - w/2,
    cycle_df["uncond_delta"],
    yerr=cycle_df["uncond_std"],
    width=w,
    label="unconditional",
)

ax.bar(
    x + w/2,
    cycle_df["cond_delta"],
    yerr=cycle_df["cond_std"],
    width=w,
    label="conditional",
)

ax.axhline(0, linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(
    cycle_df["label"],
    rotation=90,
    fontsize=8,
)

ax.set_ylabel("Δ hard NLL vs classical")
ax.set_title(
    "Per-cycle diffusion improvement (validation, per-window)\n"
    "Negative values are improvements"
)

ax.legend()

plt.tight_layout()
plt.show()

# ──────────────────────────────────────────────────────────────────────────
# 4. Print strongest improvements
# ──────────────────────────────────────────────────────────────────────────

print("\n── Strongest conditional improvements ─────────────")

display_cols = [
    "label",
    "uncond_delta",
    "cond_delta",
]

print(
    cycle_df[display_cols]
    .head(10)
    .to_string(index=False, float_format=lambda v: f"{v:.4f}")
)

print("\n✓ Task 59 complete")

---
## Reading the numbers, and what to do next

Three interpretive lines through the comparison table:

**Direction of the headline.** Is `classical + conditional` < `classical
alone` on validation? If yes and the margin is much larger than the K-σ
spread, the conditioning is doing useful work. If yes but inside the
K-σ spread, you have evidence of motion in the right direction but
sampling noise is a sizeable fraction of the signal — more samples
(K = 50 or 100) would tighten the comparison. If no, look at the
`uncond_delta` column: if both combined columns are *worse* than
classical, the diffusion machinery is making the density less
calibrated, not more, and the next iteration should focus on whether
the residuals are over-correcting (high `floor_fraction`).

**Conditioning specifically.** The interesting comparison is not just
combined vs. classical but combined-conditional vs. combined-
unconditional. If both columns are equally good (or equally bad), the
conditioning machinery is not earning its keep, even if residual
addition in general is helping. If the conditional column is meaningfully
better, you are measuring exactly the diffusion model's targeting
ability — the headline finding of Week 09.

**Per-window vs. per-year.** Per-window NLL is the diffusion model's
home turf — each window has its own residual, sampled at its own
conditioning. Per-year NLL aggregates lats over two windows but only
asks for one diffusion sample at the year-center conditioning, so the
diffusion model is doing less work per per-year block. Expect the
per-year improvement to be smaller than the per-window one; the gap
between them tells you how much the 6-month granularity is itself
contributing to the value-add.

**What about the test set?** The test split has been deliberately
untouched throughout this notebook. After you finish all model selection
(prompt iteration, hyperparameter search, architecture changes —
whatever the experimentation week brings), the test set gets opened
*once*, the same comparison table gets computed *once*, and that table
is the program's final scoreboard. Touching the test set before that
final measurement is the most common way real ML projects accidentally
inflate their reported performance, and the discipline of leaving it
alone is — in the long run — more valuable than any specific
architectural choice we've made over the last three weeks.

**The test set will only be used at the very, very end so don't use it for anything until told otherwise.**


